In [35]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import pandas as pd
import os

# Data Warehouse Connection

In [13]:
# 1. Initialize Spark Session with the Oracle JDBC Driver
# The 'spark.jars.packages' config tells Spark to automatically download the Oracle driver
spark = SparkSession.builder \
    .appName("Ecommerce_DWH_ETL") \
    .config("spark.jars.packages", "com.oracle.database.jdbc:ojdbc8:21.9.0.0") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


In [15]:
# 2. Database Connection Properties (Matching your successful terminal login)
db_user = "pdb_admin"
db_password = "rzk1234" 

# Notice the "jdbc:oracle:thin:" prefix - this is required for Spark!
jdbc_url = "jdbc:oracle:thin:@//localhost:1539/dwh_pdb"

connection_properties = {
    "user": db_user,
    "password": db_password,
    "driver": "oracle.jdbc.driver.OracleDriver"
}

In [16]:
# 3. Extract Data
try:
    fact_df = spark.read.jdbc(
        url=jdbc_url, 
        table="FACT_ORDER_LINE", 
        properties=connection_properties
    )
    print("Successfully connected to Oracle via PySpark!")
    fact_df.show(5)
except Exception as e:
    print(f"Connection failed: {e}")

Successfully connected to Oracle via PySpark!
+--------------+-------------------+-------------+-------------+-------------+-------------+-------------+--------+------------+---------------+----------+-----------+-------------+
|ORDER_LINE_KEY|           DATE_KEY| CUSTOMER_KEY|  PRODUCT_KEY| CATEGORY_KEY|  PAYMENT_KEY| SHIPPING_KEY|QUANTITY|GROSS_AMOUNT|DISCOUNT_AMOUNT|NET_AMOUNT|COST_AMOUNT|PROFIT_AMOUNT|
+--------------+-------------------+-------------+-------------+-------------+-------------+-------------+--------+------------+---------------+----------+-----------+-------------+
| 11.0000000000|20240119.0000000000|32.0000000000|32.0000000000|17.0000000000|12.0000000000|12.0000000000|       4|     1274.13|          16.18|   1257.95|     764.48|       493.47|
| 12.0000000000|20240110.0000000000|25.0000000000|33.0000000000|17.0000000000|12.0000000000|11.0000000000|       5|      484.52|          45.74|    438.78|     290.71|       148.07|
| 13.0000000000|20240220.0000000000|25.00000

## KPI Data Marts

In [31]:
print("1. Extracting Dimension Tables from Oracle...")
# We need to load DIM_DATE from Oracle to join it with our Fact table
dim_date_df = spark.read.jdbc(
    url=jdbc_url, 
    table="DIM_DATE", 
    properties=connection_properties
)

print("2. Calculating Core Business KPIs in PySpark...")

# Join Fact with Dim_Date to get month and year info
enriched_df = fact_df.join(dim_date_df, "DATE_KEY", "left")

# ==========================================
# Overall Scalar KPIs (Revenue, Profit, Margin, AOV, Repeat Rate)
# ==========================================

# Total Revenue & Gross Profit
totals = fact_df.agg(
    F.sum("NET_AMOUNT").alias("TOTAL_REVENUE"),
    F.sum("PROFIT_AMOUNT").alias("TOTAL_PROFIT")
).collect()[0]

total_revenue = totals["TOTAL_REVENUE"] or 0
total_profit = totals["TOTAL_PROFIT"] or 0

# Profit Margin
profit_margin_pct = round((total_profit / total_revenue) * 100, 2) if total_revenue > 0 else 0

# Average Order Value (AOV)
order_baskets = fact_df.groupBy("CUSTOMER_KEY", "DATE_KEY").agg(
    F.sum("NET_AMOUNT").alias("ORDER_REVENUE")
)
aov = round(order_baskets.select(F.avg("ORDER_REVENUE")).collect()[0][0], 2)

# Repeat Purchase Rate
customer_order_counts = fact_df.groupBy("CUSTOMER_KEY").agg(
    F.countDistinct("DATE_KEY").alias("UNIQUE_PURCHASE_DAYS")
)
total_customers = customer_order_counts.count()
repeat_customers = customer_order_counts.filter(F.col("UNIQUE_PURCHASE_DAYS") > 1).count()
repeat_purchase_rate = round((repeat_customers / total_customers) * 100, 2) if total_customers > 0 else 0

overall_kpis_pdf = pd.DataFrame([{
    "Total_Revenue": total_revenue,
    "Gross_Profit": total_profit,
    "Profit_Margin_Pct": profit_margin_pct,
    "AOV": aov,
    "Repeat_Purchase_Rate_Pct": repeat_purchase_rate
}])

1. Extracting Dimension Tables from Oracle...
2. Calculating Core Business KPIs in PySpark...


In [41]:
# ==========================================
# Customer Lifetime Value (CLV) Data Mart
# ==========================================
print("3. Calculating Customer Lifetime Value...")
clv_df = fact_df.groupBy("CUSTOMER_KEY").agg(
    F.round(F.sum("NET_AMOUNT"), 2).alias("CLV"),
    F.countDistinct("DATE_KEY").alias("TOTAL_ORDERS")
).orderBy(F.desc("CLV"))

3. Calculating Customer Lifetime Value...


In [40]:
# ==========================================
# Revenue Growth Rate (Month-over-Month)
# ==========================================
print("4. Calculating MoM Revenue Growth...")
monthly_revenue_df = enriched_df.groupBy("YEAR", "MONTH", "MONTH_NAME").agg(
    F.sum("NET_AMOUNT").alias("CURRENT_MONTH_REVENUE")
)

window_spec = Window.orderBy("YEAR", "MONTH")

growth_df = monthly_revenue_df.withColumn(
    "PREV_MONTH_REVENUE", F.lag("CURRENT_MONTH_REVENUE", 1).over(window_spec)
).withColumn(
    "MOM_GROWTH_PCT", 
    F.round(((F.col("CURRENT_MONTH_REVENUE") - F.col("PREV_MONTH_REVENUE")) / F.col("PREV_MONTH_REVENUE")) * 100, 2)
).orderBy("YEAR", "MONTH")

4. Calculating MoM Revenue Growth...


In [43]:
# ==========================================
# Exporting to Data Marts
# ==========================================
print("5. Exporting to Data Marts...")
output_dir = "../data"
os.makedirs(output_dir, exist_ok=True)

overall_kpis_pdf.to_csv(f"{output_dir}/overall_kpis.csv", index=False)
clv_df.toPandas().to_csv(f"{output_dir}/customer_clv.csv", index=False)
growth_df.toPandas().to_csv(f"{output_dir}/monthly_growth.csv", index=False)

5. Exporting to Data Marts...


In [44]:
print(f"Success! KPI Data Marts saved to {output_dir}/")
print("\n--- Overall Business KPIs ---")
print(overall_kpis_pdf.to_string(index=False))

Success! KPI Data Marts saved to ../data/

--- Overall Business KPIs ---
Total_Revenue Gross_Profit Profit_Margin_Pct    AOV  Repeat_Purchase_Rate_Pct
     39797.81     15157.41             38.09 765.34                     100.0
